# Lingi7 Catalog Enrichment — Full Workflow Test (Colab)

Runs the **entire FastAPI backend** on Google Colab and tests all workflows.

**Prerequisites:** Runtime > Change runtime type > **T4 GPU**

---

| Workflow | Endpoint | Status on Colab |
|----------|----------|----------------|
| VLM Analysis | POST /vlm/analyze | Works |
| FAQ Generation | POST /vlm/faqs | Works |
| Protocol Schemas (ACP/UCP) | POST /protocols/generate | Works |
| Manual PDF Extraction | POST /vlm/manual/extract | Works |
| Policy Compliance | POST /vlm/analyze (with policies) | Partial (no Milvus) |
| Image Variation (FLUX) | POST /generate/variation | Skipped (needs GPU) |
| 3D Asset (TRELLIS) | POST /generate/3d | Skipped (needs GPU) |

## 1. Install Ollama

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1 && echo 'zstd installed'
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Pull quantized models (fits in T4 15GB VRAM)

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Ollama server started!")

!ollama pull qwen2.5:7b
!ollama pull llava:7b
print("Models pulled!")
!ollama list

## 3. Install Python dependencies

In [ ]:
!pip install fastapi uvicorn openai pydantic python-multipart aiofiles pillow python-dotenv httpx pyyaml pypdf numpy pyngrok -q

## 4. Clone the repo and set up the backend

In [ ]:
import os

os.chdir("/content")

# Uncomment ONE of these:
# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/lingi7_scaffold.git
# os.chdir("/content/lingi7_scaffold/enrichment")

# Option B: Upload the enrichment folder manually via Colab file upload
# After uploading, uncomment and adjust:
# os.chdir("/content/enrichment")

# Verify we are in the right place
print(f"Current directory: {os.getcwd()}")
print(f"Backend exists: {os.path.exists('src/backend/main.py')}")

## 5. Write Colab config

In [ ]:
import os
os.makedirs("/content/enrichment/shared/config", exist_ok=True)

config = """vlm:
  url: "http://localhost:11434/v1"
  model: "llava:7b"
  max_tokens: 1024
  temperature: 0.1

llm:
  url: "http://localhost:11434/v1"
  model: "qwen2.5:7b"
  max_tokens: 2048
  temperature: 0.3

embeddings:
  url: "http://localhost:11434/v1"
  model: "qwen2.5:7b"

flux:
  url: "http://localhost:8003/v1/infer"

trellis:
  url: "http://localhost:8004/v1/infer"

milvus:
  host: "localhost"
  port: 19530
  collection: "policy_chunks"
  alias: "policy_library"

product_manual:
  chunk_size_words: 250
  chunk_overlap_words: 50
  top_k_per_query: 3
  min_relevance_score: 0.25

policy_library:
  storage_dir: "/content/data/policies"
  db_path: "/content/data/policies/library.db"
  top_k: 8
  min_relevance_score: 0.3
  max_policy_text_chars: 12000
  normalization_max_tokens: 2048
  classification_max_tokens: 1024
  embedding_batch_size: 128
  embedding_dim: 384

locales:
  default: "en-US"
  supported:
    - "en-US"
    - "en-GB"
    - "en-AU"
    - "en-CA"
    - "es-ES"
    - "es-MX"
    - "es-AR"
    - "es-CO"
    - "fr-FR"
    - "fr-CA"
"""

with open("/content/enrichment/shared/config/config.yaml", "w") as f:
    f.write(config)
print("Config written!")

## 6. Server already running from step 2

In [ ]:
print("Server already running from step 2.")

## 7. Patch policy_library.py (graceful Milvus fallback)

Milvus needs Docker and will not run on Colab free tier. This patch makes the app start cleanly and skip vector search.

In [ ]:
patch_code = """
import hashlib
import json
import logging
import os
import shutil
import sqlite3
import time
from pathlib import Path
from typing import Any, Dict, List, Sequence

from openai import OpenAI

from backend.config import get_config
from backend.policy import extract_text_from_pdf_bytes, summarize_policy_document

logger = logging.getLogger("catalog_enrichment.policy_library")

EMBEDDING_API_KEY_ERROR = "API_KEY or LLM_API_KEY is not set"
MAX_QUERY_WORDS = 160
MAX_EMBED_TOTAL_WORDS = 190
MILVUS_AVAILABLE = False

try:
    from pymilvus import Collection, CollectionSchema, DataType, FieldSchema, connections, utility
    MILVUS_AVAILABLE = True
except ImportError:
    logger.warning("pymilvus not available; vector search disabled")


def _limit_words(text, max_words):
    words = text.split()
    if len(words) <= max_words:
        return text.strip()
    return " ".join(words[:max_words]).strip()


def build_policy_query(product_snapshot):
    parts = [
        "Title: " + product_snapshot.get("title", ""),
        "Description: " + product_snapshot.get("description", ""),
        "Categories: " + ", ".join(product_snapshot.get("categories", [])),
        "Tags: " + ", ".join(product_snapshot.get("tags", [])),
        "Colors: " + ", ".join(product_snapshot.get("colors", [])),
    ]
    return _limit_words("\n".join(p for p in parts if p.strip()), MAX_QUERY_WORDS)


class PolicyLibrary:
    def __init__(self):
        config = get_config()
        self._policy_config = config.get_policy_library_config()
        self._milvus_config = config.get_milvus_config()
        self._embedding_config = config.get_embeddings_config()
        self._storage_dir = Path(self._policy_config["storage_dir"])
        self._db_path = Path(self._policy_config["db_path"])
        self._top_k = int(self._policy_config["top_k"])
        self._min_relevance_score = float(self._policy_config["min_relevance_score"])
        self._collection_name = str(self._milvus_config["collection"])
        self._milvus_alias = str(self._milvus_config["alias"])
        self._connected = False

    def initialize(self):
        self._storage_dir.mkdir(parents=True, exist_ok=True)
        self._db_path.parent.mkdir(parents=True, exist_ok=True)
        with self._connect_db() as conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS policy_documents (
                document_hash TEXT PRIMARY KEY, filename TEXT NOT NULL,
                file_size INTEGER NOT NULL, chunk_count INTEGER NOT NULL,
                summary_json TEXT NOT NULL, text_path TEXT NOT NULL,
                created_at INTEGER NOT NULL, updated_at INTEGER NOT NULL)""")
            conn.commit()

    def list_documents(self):
        with self._connect_db() as conn:
            rows = conn.execute(
                "SELECT document_hash, filename, file_size, chunk_count, created_at, updated_at "
                "FROM policy_documents ORDER BY updated_at DESC"
            ).fetchall()
        return [{"document_hash": r["document_hash"], "filename": r["filename"],
                 "file_size": r["file_size"], "chunk_count": r["chunk_count"],
                 "created_at": r["created_at"], "updated_at": r["updated_at"]} for r in rows]

    def ingest_documents(self, uploads, locale="en-US"):
        results = []
        for upload in uploads:
            filename = upload["filename"]
            pdf_bytes = upload["bytes"]
            document_hash = hashlib.sha256(pdf_bytes).hexdigest()
            existing = self._get_document(document_hash)
            if existing is not None:
                self._touch_document(document_hash)
                results.append({"document_hash": document_hash, "filename": existing["filename"],
                               "chunk_count": existing["chunk_count"], "already_loaded": True, "processed": False})
                continue
            extracted_text = extract_text_from_pdf_bytes(pdf_bytes)
            if not extracted_text:
                raise ValueError("Unable to extract text from PDF: " + filename)
            normalized_text = extracted_text.strip()
            summary = summarize_policy_document(filename, normalized_text, locale)
            records = self._build_policy_entries(filename, summary)
            if MILVUS_AVAILABLE:
                embedding_inputs = [self._format_policy_entry_for_embedding(e) for e in records]
                vectors = self._embed_texts(embedding_inputs, input_type="passage")
                if vectors:
                    self._ensure_collection(len(vectors[0]))
                    self._replace_document_vectors(document_hash, filename, summary, records, vectors)
            self._persist_document(document_hash, filename, len(pdf_bytes), len(records), summary, normalized_text)
            results.append({"document_hash": document_hash, "filename": filename,
                           "chunk_count": len(records), "already_loaded": False, "processed": True})
        return results

    def retrieve_context(self, product_snapshot):
        if not self.list_documents():
            return []
        if not MILVUS_AVAILABLE:
            return []
        if not self._collection_exists():
            return []
        query_text = build_policy_query(product_snapshot)
        if not query_text.strip():
            return []
        query_vector = self._embed_texts([query_text], input_type="query")[0]
        collection = self._get_collection(load=True)
        results = collection.search(data=[query_vector], anns_field="embedding",
            param={"metric_type": "COSINE", "params": {}}, limit=self._top_k,
            output_fields=["document_hash", "document_name", "policy_title", "summary", "chunk_text", "chunk_index"])
        raw_hits = []
        document_hashes = set()
        for hit in results[0]:
            entity = hit.entity
            document_hash = entity.get("document_hash")
            if document_hash:
                document_hashes.add(document_hash)
            raw_hits.append((hit, entity))
        if raw_hits:
            top_score = float(raw_hits[0][0].score)
            if top_score < self._min_relevance_score:
                return []
        document_summaries = self._get_document_summaries(document_hashes)
        retrieved = []
        for hit, entity in raw_hits:
            document_hash = entity.get("document_hash")
            retrieved.append({"document_hash": document_hash, "document_name": entity.get("document_name"),
                "policy_title": entity.get("policy_title"), "summary": entity.get("summary"),
                "chunk_text": entity.get("chunk_text"), "chunk_index": entity.get("chunk_index"),
                "score": float(hit.score), "document_summary": document_summaries.get(document_hash, {})})
        return retrieved

    def clear(self):
        if MILVUS_AVAILABLE and self._collection_exists():
            utility.drop_collection(self._collection_name, using=self._milvus_alias)
        if self._db_path.exists():
            with self._connect_db() as conn:
                conn.execute("DELETE FROM policy_documents")
                conn.commit()
        if self._storage_dir.exists():
            for child in self._storage_dir.iterdir():
                if child.is_dir():
                    shutil.rmtree(child)
                else:
                    child.unlink()
        self._storage_dir.mkdir(parents=True, exist_ok=True)
        self._db_path.parent.mkdir(parents=True, exist_ok=True)
        self.initialize()

    def _connect_db(self):
        conn = sqlite3.connect(self._db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _get_document(self, document_hash):
        with self._connect_db() as conn:
            row = conn.execute("SELECT document_hash, filename, chunk_count FROM policy_documents WHERE document_hash = ?", (document_hash,)).fetchone()
        return row

    def _touch_document(self, document_hash):
        now = int(time.time())
        with self._connect_db() as conn:
            conn.execute("UPDATE policy_documents SET updated_at = ? WHERE document_hash = ?", (now, document_hash))
            conn.commit()

    def _persist_document(self, document_hash, filename, file_size, chunk_count, summary, extracted_text):
        now = int(time.time())
        document_dir = self._storage_dir / document_hash
        document_dir.mkdir(parents=True, exist_ok=True)
        (document_dir / "text.txt").write_text(extracted_text, encoding="utf-8")
        (document_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        with self._connect_db() as conn:
            conn.execute("INSERT OR REPLACE INTO policy_documents (document_hash, filename, file_size, chunk_count, summary_json, text_path, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                (document_hash, filename, file_size, chunk_count, json.dumps(summary, ensure_ascii=False), str(document_dir / "text.txt"), now, now))
            conn.commit()

    def _get_document_summaries(self, document_hashes):
        unique = [h for h in dict.fromkeys(document_hashes) if h]
        if not unique:
            return {}
        placeholders = ",".join("?" for _ in unique)
        with self._connect_db() as conn:
            rows = conn.execute("SELECT document_hash, summary_json FROM policy_documents WHERE document_hash IN (" + placeholders + ")", unique).fetchall()
        summaries = {}
        for row in rows:
            try:
                summaries[row["document_hash"]] = json.loads(row["summary_json"])
            except json.JSONDecodeError:
                summaries[row["document_hash"]] = {}
        return summaries

    def _build_policy_entries(self, filename, summary):
        entries = []
        overview_lines = ["Document: " + filename, "Policy Title: " + summary.get("policy_title", filename),
            "Summary: " + summary.get("summary", ""), "Rule Type: Overview"]
        for item in summary.get("required_evidence", []):
            if str(item).strip():
                overview_lines.append("Required Evidence: " + str(item))
        entries.append("\n".join(overview_lines))
        for rule_type, rules in ("Blocking", summary.get("blocking_rules", [])), ("Permitted", summary.get("permitted_rules", [])):
            for rule in rules:
                lines = ["Document: " + filename, "Policy Title: " + summary.get("policy_title", filename),
                    "Rule Type: " + rule_type, "Rule Title: " + rule.get("title", ""),
                    "Conditions: " + "; ".join(str(i) for i in rule.get("conditions", []))]
                entries.append("\n".join(l for l in lines if l.strip()))
        return [_limit_words(e, MAX_EMBED_TOTAL_WORDS) for e in entries if e.strip()]

    def _format_policy_entry_for_embedding(self, text):
        return _limit_words(text, MAX_EMBED_TOTAL_WORDS)

    def _embed_texts(self, texts, input_type):
        if not texts:
            return []
        api_key = os.getenv("API_KEY", os.getenv("LLM_API_KEY", ""))
        if not api_key:
            raise RuntimeError(EMBEDDING_API_KEY_ERROR)
        client = OpenAI(api_key=api_key, base_url=self._embedding_config["url"])
        response = client.embeddings.create(input=list(texts), model=self._embedding_config["model"],
            encoding_format="float", extra_body={"input_type": input_type, "truncate": "NONE"})
        return [item.embedding for item in response.data]

    def _connect_milvus(self):
        if not MILVUS_AVAILABLE or self._connected:
            return
        try:
            connections.connect(alias=self._milvus_alias, host=self._milvus_config["host"], port=self._milvus_config["port"])
            self._connected = True
        except Exception as e:
            logger.warning("Milvus connection failed: %s", e)

    def _collection_exists(self):
        self._connect_milvus()
        if not MILVUS_AVAILABLE or not self._connected:
            return False
        return utility.has_collection(self._collection_name, using=self._milvus_alias)

    def _ensure_collection(self, dimension):
        self._connect_milvus()
        if not MILVUS_AVAILABLE or not self._connected:
            return
        if utility.has_collection(self._collection_name, using=self._milvus_alias):
            return
        fields = [
            FieldSchema(name="chunk_id", dtype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=128),
            FieldSchema(name="document_hash", dtype=DataType.VARCHAR, max_length=64),
            FieldSchema(name="document_name", dtype=DataType.VARCHAR, max_length=512),
            FieldSchema(name="policy_title", dtype=DataType.VARCHAR, max_length=512),
            FieldSchema(name="summary", dtype=DataType.VARCHAR, max_length=4096),
            FieldSchema(name="chunk_text", dtype=DataType.VARCHAR, max_length=16384),
            FieldSchema(name="chunk_index", dtype=DataType.INT64),
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dimension),
        ]
        schema = CollectionSchema(fields=fields, description="Policy records")
        collection = Collection(name=self._collection_name, schema=schema, using=self._milvus_alias)
        collection.create_index("embedding", {"index_type": "AUTOINDEX", "metric_type": "COSINE", "params": {}})
        collection.load()

    def _get_collection(self, load=False):
        self._connect_milvus()
        collection = Collection(name=self._collection_name, using=self._milvus_alias)
        if load:
            collection.load()
        return collection

    def _replace_document_vectors(self, document_hash, filename, summary, records, vectors):
        collection = self._get_collection(load=True)
        collection.delete(expr='document_hash == "' + document_hash + '"')
        entities = [
            [document_hash + ":" + str(i) for i in range(len(records))],
            [document_hash] * len(records), [filename] * len(records),
            [str(summary.get("policy_title", filename))] * len(records),
            [str(summary.get("summary", ""))] * len(records),
            list(records), list(range(len(records))), [list(v) for v in vectors],
        ]
        collection.insert(entities)
        collection.flush()
"""

with open("/content/enrichment/src/backend/policy_library.py", "w") as f:
    f.write(patch_code)
print("policy_library.py patched for Colab!")

## 8. Start the FastAPI server

In [ ]:
import os
os.environ["API_KEY"] = "ollama"
os.environ["LLM_API_KEY"] = "ollama"

import subprocess, time, sys

!pkill -f "uvicorn.*backend.main" 2>/dev/null || true
time.sleep(1)

server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd="/content/enrichment/src",
)

time.sleep(5)
if server_proc.poll() is not None:
    print("Server failed to start. Output:")
    print(server_proc.stdout.read().decode())
else:
    print(f"Server started (PID {server_proc.pid}) on port 8000")

## 9. Expose via ngrok tunnel

In [ ]:
from pyngrok import ngrok

tunnel = ngrok.connect(8000)
BASE_URL = tunnel.public_url
print(f"Backend URL: {BASE_URL}")

## 10. Health check

In [ ]:
import requests

r = requests.get(f"{BASE_URL}/health")
print(f"Health: {r.status_code} - {r.json()}")

r = requests.get(f"{BASE_URL}/")
print(f"Homepage: {r.status_code} - {r.text}")

## 11. Upload a product image

In [ ]:
from google.colab import files
from PIL import Image
import io

print("Upload a product image (JPEG/PNG):")
uploaded = files.upload()

if not uploaded:
    print("No image uploaded. Creating a sample...")
    img = Image.new("RGB", (400, 400), color=(255, 100, 50))
    img.save("/content/sample_product.jpg", "JPEG")
    IMAGE_BYTES = open("/content/sample_product.jpg", "rb").read()
    CONTENT_TYPE = "image/jpeg"
else:
    filename = list(uploaded.keys())[0]
    img = Image.open(io.BytesIO(uploaded[filename]))
    print(f"Image loaded: {img.size}, mode: {img.mode}")
    IMAGE_BYTES = uploaded[filename]
    if hasattr(uploaded[filename], 'headers'):
        CONTENT_TYPE = uploaded[filename].headers.get("content-type", "image/jpeg")
    else:
        CONTENT_TYPE = "image/jpeg"
    if "png" in filename.lower():
        CONTENT_TYPE = "image/png"
    elif "jpg" in filename.lower() or "jpeg" in filename.lower():
        CONTENT_TYPE = "image/jpeg"

print(f"Image ready: {len(IMAGE_BYTES)} bytes, content_type={CONTENT_TYPE}")

## 12. Test: VLM Analysis (POST /vlm/analyze)

Analyzes the product image and returns enriched catalog fields.

In [ ]:
import json, time

print("Running VLM analysis (this may take 30-60s on T4)...")
start = time.time()

r = requests.post(
    f"{BASE_URL}/vlm/analyze",
    files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
    data={"locale": "en-US"},
    timeout=180,
)

elapsed = time.time() - start
print(f"Status: {r.status_code} ({elapsed:.1f}s)")

if r.status_code == 200:
    vlm_result = r.json()
    print(json.dumps(vlm_result, indent=2, ensure_ascii=False))
else:
    print(f"Error: {r.text}")
    vlm_result = None

## 13. Test: FAQ Generation (POST /vlm/faqs)

Generates 3-5 product FAQs from the enriched data.

In [ ]:
if vlm_result:
    print("Generating FAQs...")
    start = time.time()

    r = requests.post(
        f"{BASE_URL}/vlm/faqs",
        data={
            "title": vlm_result.get("title", ""),
            "description": vlm_result.get("description", ""),
            "categories": json.dumps(vlm_result.get("categories", [])),
            "tags": json.dumps(vlm_result.get("tags", [])),
            "colors": json.dumps(vlm_result.get("colors", [])),
            "locale": "en-US",
        },
        timeout=120,
    )

    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")

    if r.status_code == 200:
        faqs_result = r.json()
        print(json.dumps(faqs_result, indent=2, ensure_ascii=False))
    else:
        print(f"Error: {r.text}")
        faqs_result = None
else:
    print("Skipping - no VLM result available")
    faqs_result = None

## 14. Test: Protocol Schema Export (POST /protocols/generate)

Generates both ACP and UCP commerce protocol schemas.

In [ ]:
if vlm_result:
    print("Generating ACP/UCP protocol schemas...")
    start = time.time()

    faqs_list = faqs_result.get("faqs", []) if faqs_result else []

    r = requests.post(
        f"{BASE_URL}/protocols/generate",
        data={
            "title": vlm_result.get("title", ""),
            "description": vlm_result.get("description", ""),
            "categories": json.dumps(vlm_result.get("categories", [])),
            "tags": json.dumps(vlm_result.get("tags", [])),
            "colors": json.dumps(vlm_result.get("colors", [])),
            "faqs": json.dumps(faqs_list),
            "locale": "en-US",
        },
        timeout=120,
    )

    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")

    if r.status_code == 200:
        protocols_result = r.json()
        print("ACP keys:", list(protocols_result.get("acp", {}).keys()))
        print("UCP keys:", list(protocols_result.get("ucp", {}).keys()))
        print("ACP title:", protocols_result.get("acp", {}).get("product", {}).get("title", "")[:80])
    else:
        print(f"Error: {r.text}")
else:
    print("Skipping - no VLM result available")

## 15. Test: Manual PDF Extraction (POST /vlm/manual/extract)

Upload a product manual PDF to extract knowledge for richer FAQ generation.

In [ ]:
print("Upload a product manual PDF (optional):")
uploaded_pdf = files.upload()

manual_result = None
if uploaded_pdf:
    pdf_name = list(uploaded_pdf.keys())[0]
    pdf_bytes = uploaded_pdf[pdf_name]
    if hasattr(pdf_bytes, 'read'):
        pdf_bytes = pdf_bytes.read()

    print(f"PDF uploaded: {pdf_name} ({len(pdf_bytes)} bytes)")
    print("Extracting knowledge from manual...")
    start = time.time()

    r = requests.post(
        f"{BASE_URL}/vlm/manual/extract",
        files={"file": (pdf_name, pdf_bytes, "application/pdf")},
        data={
            "title": vlm_result.get("title", "") if vlm_result else "",
            "categories": json.dumps(vlm_result.get("categories", [])) if vlm_result else "[]",
            "locale": "en-US",
        },
        timeout=120,
    )

    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")

    if r.status_code == 200:
        manual_result = r.json()
        print(f"Filename: {manual_result.get('filename')}")
        print(f"Chunks: {manual_result.get('chunk_count')}")
        knowledge = manual_result.get("knowledge", {})
        print(f"Knowledge topics: {list(knowledge.keys())}")
        for topic, text in knowledge.items():
            preview = text[:120] + "..." if len(text) > 120 else text
            print(f"  [{topic}]: {preview}")
    else:
        print(f"Error: {r.text}")
else:
    print("No PDF uploaded - skipping manual extraction test.")

## 16. Test: FAQ Generation with Manual Knowledge

In [ ]:
if vlm_result and manual_result and manual_result.get("knowledge"):
    print("Generating enriched FAQs with manual knowledge...")
    start = time.time()

    r = requests.post(
        f"{BASE_URL}/vlm/faqs",
        data={
            "title": vlm_result.get("title", ""),
            "description": vlm_result.get("description", ""),
            "categories": json.dumps(vlm_result.get("categories", [])),
            "tags": json.dumps(vlm_result.get("tags", [])),
            "colors": json.dumps(vlm_result.get("colors", [])),
            "locale": "en-US",
            "manual_knowledge": json.dumps(manual_result["knowledge"]),
        },
        timeout=120,
    )

    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")

    if r.status_code == 200:
        enriched_faqs = r.json()
        print(json.dumps(enriched_faqs, indent=2, ensure_ascii=False))
    else:
        print(f"Error: {r.text}")
else:
    print("Skipping - need both VLM result and manual knowledge")

## 17. Test: Policy Status

In [ ]:
r = requests.get(f"{BASE_URL}/policies")
print(f"Loaded policies: {r.json()}")

print("Note: Policy compliance check ran automatically during /vlm/analyze.")
if vlm_result and "policy_decision" in vlm_result:
    print(f"Policy decision: {json.dumps(vlm_result['policy_decision'], indent=2)}")
else:
    print("No policy_decision in VLM result (no policies were loaded).")

## 18. Test: VLM with Augmentation (existing product data)

In [ ]:
if IMAGE_BYTES:
    print("Running VLM analysis with product data augmentation...")
    start = time.time()

    r = requests.post(
        f"{BASE_URL}/vlm/analyze",
        files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
        data={
            "locale": "en-US",
            "product_data": json.dumps({
                "title": "Blue Cotton T-Shirt",
                "description": "A comfortable everyday t-shirt made from 100% organic cotton.",
                "price": 29.99,
                "sku": "TSH-BLU-001",
            }),
            "brand_instructions": "Use a friendly, casual tone. Emphasize sustainability and comfort.",
        },
        timeout=180,
    )

    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")

    if r.status_code == 200:
        augmented_result = r.json()
        print(json.dumps(augmented_result, indent=2, ensure_ascii=False))
    else:
        print(f"Error: {r.text}")

## 19. Test: Multi-locale Analysis (es-MX, fr-FR)

In [ ]:
if IMAGE_BYTES:
    for locale in ["es-MX", "fr-FR"]:
        print(f"\n--- Testing locale: {locale} ---")
        start = time.time()

        r = requests.post(
            f"{BASE_URL}/vlm/analyze",
            files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
            data={"locale": locale},
            timeout=180,
        )

        elapsed = time.time() - start
        if r.status_code == 200:
            result = r.json()
            print(f"Title: {result.get('title', '')[:80]}")
            print(f"Categories: {result.get('categories', [])}")
        else:
            print(f"Error: {r.status_code} - {r.text[:200]}")

## 20. Test: Invalid inputs (error handling)

In [ ]:
print("--- Test: Invalid locale ---")
r = requests.post(
    f"{BASE_URL}/vlm/analyze",
    files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
    data={"locale": "xx-XX"},
)
print(f"Status: {r.status_code} - {r.json()}\n")

print("--- Test: Empty image ---")
r = requests.post(
    f"{BASE_URL}/vlm/analyze",
    files={"image": ("empty.jpg", b"", "image/jpeg")},
    data={"locale": "en-US"},
)
print(f"Status: {r.status_code} - {r.json()}\n")

print("--- Test: Non-image file ---")
r = requests.post(
    f"{BASE_URL}/vlm/analyze",
    files={"image": ("test.txt", b"hello", "text/plain")},
    data={"locale": "en-US"},
)
print(f"Status: {r.status_code} - {r.json()}\n")

print("--- Test: Invalid JSON in product_data ---")
r = requests.post(
    f"{BASE_URL}/vlm/analyze",
    files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
    data={"locale": "en-US", "product_data": "not-json"},
)
print(f"Status: {r.status_code} - {r.json()}")

## Summary

### Worked on Colab (T4 GPU)
- VLM Analysis - Image to product data
- LLM Enhancement - Richer copywriting
- Brand Alignment - Brand voice overlay
- FAQ Generation - 3-5 product FAQs
- Manual Knowledge Extraction - PDF to structured knowledge
- Enriched FAQ Generation - FAQs with manual knowledge
- Protocol Schema Export - ACP and UCP schemas
- Multi-locale Support - en-US, es-MX, fr-FR
- Augmentation Mode - Merge existing product data
- Error Handling - Invalid inputs handled gracefully

### Partially working
- Policy Compliance - SQLite ingestion works, but vector search needs Milvus (Docker)

### Not available on Colab free tier
- Image Variation (FLUX) - Needs 24GB+ VRAM
- 3D Asset Generation (TRELLIS) - Needs 24GB+ VRAM